# Proteomics Data Visualization
**DDEA Proteomics Course — Hands-On Tutorial**

> **Before starting:**
> 1. Go to **Runtime → Change runtime type** and select **R**.
> 2. Run **Cell 1** (Mount Drive). If prompted to authorise, click the link, sign in, then re-run Cell 1.
> 3. Run **Cell 2** (Install / load packages). First run takes ~5–10 min; subsequent sessions load from Drive in seconds.

In [ ]:
# === Cell 1: Mount Google Drive ===
# Run this first. If prompted to authorise, click the link, sign in, then re-run.
system("python3 -c \"from google.colab import drive; drive.mount('/content/drive')\"")

LIB_DIR <- '/content/drive/MyDrive/DDEA_proteomics_R_libs'
dir.create(LIB_DIR, recursive = TRUE, showWarnings = FALSE)
.libPaths(c(LIB_DIR, .libPaths()))

cat('Drive mounted. Library folder:', LIB_DIR, '\n')
cat('Packages already installed:', length(list.dirs(LIB_DIR, recursive = FALSE)), '\n')

In [ ]:
# === Cell 2: Install or load packages ===
# Safe to re-run — skips packages already in your Drive library.

LIB_DIR <- '/content/drive/MyDrive/DDEA_proteomics_R_libs'
.libPaths(c(LIB_DIR, .libPaths()))

options(repos = c(PPM = 'https://packagemanager.posit.co/cran/__linux__/jammy/latest'))

if (!requireNamespace('pak', quietly = TRUE)) {
  install.packages('pak', lib = LIB_DIR)
}

all_pkgs <- c(
  'dplyr', 'forcats', 'ggplot2', 'ggplotify', 'pheatmap', 'plotly',
  'purrr', 'scales', 'stringr', 'tibble', 'tidyr', 'tidyverse', 'Seurat',
  'bioc::BiocParallel', 'bioc::clusterProfiler', 'bioc::limma',
  'bioc::miloR', 'bioc::org.Hs.eg.db', 'bioc::PhosR', 'bioc::SingleCellExperiment',
  'aj-grant/navmix'
)

pkg_base <- function(spec) sub('^.+/', '', sub('^bioc::', '', spec))

installed_pkgs <- rownames(installed.packages(lib.loc = LIB_DIR))
new_pkgs <- all_pkgs[!sapply(all_pkgs, function(p) pkg_base(p) %in% installed_pkgs)]

if (length(new_pkgs) == 0) {
  cat('All packages already installed.\n')
} else {
  cat('Installing', length(new_pkgs), 'package(s) in parallel via pak:\n')
  cat(paste(' -', new_pkgs, collapse = '\n'), '\n\n')
  pak::pak(new_pkgs, lib = LIB_DIR, ask = FALSE)
}

# Clone course data using absolute path so re-runs work correctly
if (!dir.exists('/content/DDEA_proteomics_course'))
  system('git clone --depth 1 https://github.com/fpm-cbmr/DDEA_proteomics_course.git /content/DDEA_proteomics_course')
setwd('/content/DDEA_proteomics_course')

cat('\nReady. Working directory:', getwd(), '\n')
cat('data/:\n');    print(list.files('data'))
cat('results/:\n'); print(list.files('results'))

In [ ]:
library(Seurat)
library(clusterProfiler)
library(org.Hs.eg.db)
library(limma)
library(plotly)
library(pheatmap)
library(miloR)
library(navmix)
library(tidyverse)


---

## Learning Objectives

By the end of this session you will be able to:

1. Describe the structure of a **large-scale proteomics dataset**, access its intensity matrix and sample metadata, and distinguish biological from technical covariates.
2. **Run PCA and UMAP on protein-intensity data**, interpret scree plots and embedding layouts, and explain how PC scores feed into the UMAP neighbourhood graph.
3. **Use PC loadings to run a ranked GSEA and identify biological processes driving each principal component**; relate PC scores to experimental covariates using linear models.
4. Assign biological labels to samples using canonical marker proteins and Seurat's graph-based clustering, then **apply MILO neighbourhood testing to localise case/control differences in the high-dimensional space** across three complementary contrasts.
5. **Interpret differential abundance results using boxplots, interactive volcano plots, UMAP feature overlays, and intensity–PC1 scatter plots** with GAM smoothers, distinguishing main effects from covariate-type-specific interactions.
6. **Build heatmaps of model-derived log fold changes**, **cluster proteins into co-regulated modules** using navmix, and **run GO Biological Process over-representation analysis to link each module to a biological function**.

---

# Dataset: Human Single Muscle Fiber Proteomics

The example dataset contains protein intensities for **thousands of single muscle fibers** dissected from human donors before and after an 8-week exercise training intervention. The data is distributed as an `.rds` file containing a **named list** with two objects:

| Object | Type | Rows | Columns | Notes |
|---|---|---|---|---|
| `intensities` | `matrix` / `data.frame` | proteins | single fibers | log2-transformed, normalized intensities |
| `metadata` | `data.frame` | single fibers | sample-level annotations | fiber ID, donor ID, exercise time (pre or post), and fiber number. |

In [ ]:
fiber_data  <- readRDS('data/single_fiber_proteomics_simulated.rds')
str(fiber_data, max.level = 1)

intensities <- fiber_data$intensities   # proteins x fibers
metadata    <- fiber_data$metadata      # one row per fiber

dim(intensities)
head(metadata)

---

# Stage 1: Dimensionality Reduction & Clustering

> **📊 Section overview:** Raw protein intensity matrix → **PCA** (linear; gives PC scores per fiber and loadings per protein) → **UMAP** (non-linear; uses PC scores to build a 2-D fiber-state map) → **Louvain clustering** (assigns fiber-type labels fast/slow using MYH7/MYH2) → **MILO** (neighbourhood differential abundance on the same PCA/UMAP, three contrasts: main time effect, fast-fiber only, slow-fiber only).

![Analysis flow for Stage 1](https://raw.githubusercontent.com/fpm-cbmr/DDEA_proteomics_course/master/images/mermaid_stage_1_workflow.png)

*Boxes with rounded corners are methods; rectangles are data objects; shaded subgraphs group the subsections.*

In [ ]:
# prcomp expects observations in rows → transpose: rows = fibers, columns = proteins
# PhosR::tImpute handles missing values before PCA
imp <- intensities |> PhosR::tImpute() |> as.data.frame()

pca_res <- prcomp(t(imp), scale. = TRUE)

# Scree plot: % variance explained per PC
var_exp <- (pca_res$sdev^2) / sum(pca_res$sdev^2)

tibble(PC = seq_along(var_exp), var_exp = var_exp) |>
  slice_head(n = 20) |>
  ggplot(aes(PC, var_exp)) +
  geom_col() +
  scale_x_continuous(breaks = 1:20) +
  scale_y_continuous(labels = scales::percent) +
  theme_minimal() +
  labs(y = 'Variance explained')

# Attach PC1 & PC2 to metadata for downstream use
metadata <- metadata |>
  left_join(
    as_tibble(pca_res$x, rownames = 'fiber_id') |> select(fiber_id, PC1, PC2),
    by = 'fiber_id'
  )

# PCA scatter: PC1 vs PC2, coloured by time
ggplot(metadata, aes(PC1, PC2, colour = time)) +
  geom_point(size = 0.8, alpha = 0.7) +
  theme_minimal()


> **Exercise 1.2:** Look at the scree plot. How many PCs together explain the major part of the variance?

<details>
<summary><strong>Exercise 1.2 — possible answer</strong></summary>

PC1 alone explains a large fraction of variance — often 3× more than PC2 in this dataset. This indicates one very dominant axis (fiber type) driving most of the variation.

</details>

> **Exercise 1.3:** Time doesn't seem to be explained by either PC1 or PC2.
>
> 1. What about subject? Can either PC1 or PC2 explain subject variability? Color the PCA by subject and find out!
> 2. Try extracting PC3 and plotting it against PC1. Does it capture the effect of time better?

In [ ]:
# Your code here — Part 1: colour the PCA scatter by subject_id


<details>
<summary><strong>Exercise 1.3.1 — solution</strong></summary>

```r
ggplot(metadata, aes(PC1, PC2, colour = subject_id)) +
  geom_point(size = 0.8, alpha = 0.7) +
  theme_minimal()
```

</details>

In [ ]:
# Your code here — Part 2: extract PC3 and plot against PC1, coloured by time


<details>
<summary><strong>Exercise 1.3.2 — solution</strong></summary>

```r
metadata_exercise <- metadata %>%
  left_join(
    as_tibble(pca_res$x, rownames = 'fiber_id') %>% select(fiber_id, PC3),
    by = 'fiber_id'
  )

ggplot(metadata_exercise, aes(PC1, PC3, colour = time)) +
  geom_point(size = 0.8, alpha = 0.7) +
  theme_minimal()
```

</details>

## 1.2 Extracting vectors of information from PCA

The single fiber PC scores (sample-level information) live in `pca_res$x` (one row per fiber, one column per PC). Protein loadings live in `pca_res$rotation` (feature-level information). These two matrices can be used to extract valuable information for downstream analyses:

- We already joined the PC score/sample for PC1 and PC2 back to `metadata` for downstream modelling.
- These can be tested for associations with protein intensities using linear models, GAMs, etc. (`lm( ~ intensity + PCx)`).
- By extracting the loadings for each PC, we can also see which proteins are the greatest drivers of that PC, linking PCs with biological information.

Let's first check whether some of our PCs are associated with the covariate `time`:

In [ ]:
pc_scores <- as_tibble(pca_res$x, rownames = 'fiber_id') |>
  left_join(metadata |> dplyr::select(!c(PC1, PC2)), by = 'fiber_id')

summary(lm(PC1 ~ time, data = pc_scores))
summary(lm(PC3 ~ time, data = pc_scores))


PC3 shows a stronger association with `time` than PC1. Now we use PC1 loadings as a ranked list for GSEA to find which biological processes drive the dominant fiber-type axis. GSEA functions take some time to run — run line by line and be patient.

In [ ]:
PC1_loadings <- sort(pca_res$rotation[, 'PC1'], decreasing = TRUE)
head(PC1_loadings, 10)
tail(PC1_loadings, 10)

BiocParallel::register(BiocParallel::SerialParam())
set.seed(42)
gse_PC1 <- clusterProfiler::gseGO(
  geneList     = PC1_loadings,
  OrgDb        = org.Hs.eg.db::org.Hs.eg.db,
  keyType      = 'SYMBOL',
  ont          = 'BP',
  minGSSize    = 15,
  maxGSSize    = 500,
  pvalueCutoff = 0.05,
  verbose      = FALSE
)

gse_PC1_df <- as_tibble(gse_PC1@result) |> arrange(p.adjust)
top_terms  <- bind_rows(
  gse_PC1_df |> filter(NES > 0) |> slice_head(n = 10),
  gse_PC1_df |> filter(NES < 0) |> slice_head(n = 10)
)

top_terms |>
  ggplot(aes(x = NES, y = forcats::fct_reorder(Description, NES),
             size = -log10(p.adjust), colour = NES)) +
  geom_point() +
  geom_vline(xintercept = 0, linetype = 'dashed', colour = 'grey60') +
  scale_colour_gradient2(low = 'steelblue', mid = 'grey80', high = 'firebrick', midpoint = 0) +
  theme_minimal() +
  labs(title = 'GO:BP terms ranked by PC1 loading',
       x = 'Normalized enrichment score (NES)', y = NULL, size = '-log10(p.adjust)')


> **Exercise 1.4:** Repeat the ranked GSEA for **PC2** and **PC3**. What is the dominant biology associated with each? Do any pathways appear on multiple PCs?

In [ ]:
# Your code here — extract loadings for PC2 and PC3, run gseGO, plot dot plots


<details>
<summary><strong>Exercise 1.4 — solution</strong></summary>

The workflow is identical to PC1 — only the column we pull from `pca_res$rotation` changes. You could also consider wrapping it in a small helper to avoid repeating yourself :)

```r
plot_pc_gsea <- function(pc, seed = 42) {
  loadings <- sort(pca_res$rotation[, pc], decreasing = TRUE)
  BiocParallel::register(BiocParallel::SerialParam())
  set.seed(seed)
  gse <- clusterProfiler::gseGO(
    geneList = loadings, OrgDb = org.Hs.eg.db::org.Hs.eg.db,
    keyType = 'SYMBOL', ont = 'BP',
    minGSSize = 15, maxGSSize = 500, pvalueCutoff = 0.05, verbose = FALSE
  )
  top <- as_tibble(gse@result) |> arrange(p.adjust) |>
    { bind_rows(filter(., NES > 0) |> slice_head(n = 10),
                filter(., NES < 0) |> slice_head(n = 10)) }
  ggplot(top, aes(NES, forcats::fct_reorder(Description, NES),
                  size = -log10(p.adjust), colour = NES)) +
    geom_point() +
    geom_vline(xintercept = 0, linetype = 'dashed', colour = 'grey60') +
    scale_colour_gradient2(low = 'steelblue', mid = 'grey80', high = 'firebrick', midpoint = 0) +
    theme_minimal() +
    labs(title = paste('GO:BP —', pc, 'loading'), x = 'NES', y = NULL)
}

plot_pc_gsea('PC2')
plot_pc_gsea('PC3')
```

</details>

## 1.3 Uniform Manifold Approximation and Projection (UMAP)

UMAP is a non-linear embedding that prioritizes preserving **local** neighbourhood structure. It uses multiple PCs rather than raw intensities, making it well-suited for revealing discrete populations (fast vs. slow fibers) that don't separate along a single linear axis.

We use the **Seurat workflow**: inject our `prcomp` PCs into a Seurat object so the entire pipeline (PCA → UMAP → MILO) shares a single set of coordinates.

In [ ]:
md <- metadata |> tibble::column_to_rownames('fiber_id')
seu <- Seurat::CreateSeuratObject(counts = imp, meta.data = md)
Seurat::VariableFeatures(seu) <- rownames(seu)
seu[['RNA']]$data <- seu[['RNA']]$counts

pc_emb <- pca_res$x[, 1:30]
pc_ldg <- pca_res$rotation[, 1:30]
colnames(pc_emb) <- paste0('PC_', seq_len(ncol(pc_emb)))
colnames(pc_ldg) <- paste0('PC_', seq_len(ncol(pc_ldg)))

seu[['pca']] <- Seurat::CreateDimReducObject(
  embeddings = pc_emb[colnames(seu), ],
  loadings   = pc_ldg,
  assay      = Seurat::DefaultAssay(seu),
  key        = 'PC_'
)

set.seed(42)
seu <- Seurat::RunUMAP(seu, dims = 1:20, verbose = FALSE)
Seurat::DimPlot(seu, reduction = 'umap', group.by = 'time', pt.size = 0.4)

In [ ]:
# Store UMAP coordinates for custom plots
umap_df <- Seurat::Embeddings(seu, 'umap') |>
  as_tibble(rownames = 'fiber_id') |>
  dplyr::rename(UMAP1 = umap_1, UMAP2 = umap_2) |>
  left_join(metadata, by = 'fiber_id')

> **Exercise 1.5:** Compare the UMAP to the PC1 vs PC2 and PC1 vs PC3 scatter plots. Does any structure become visible in the UMAP that the linear projections missed? Can you tell from the UMAP alone whether fibers separate by `time`?

<details>
<summary><strong>Exercise 1.5 — possible answer</strong></summary>

UMAP often resolves discrete sub-populations into well-separated islands that PCs show only as smooth gradients. The fiber-type structure (slow/fast) tends to be clearer in UMAP. The `time` effect is subtle and typically does not drive the UMAP layout — consistent with it explaining less variance than fiber type.

</details>

> **Exercise 1.6:** UMAP has two key parameters. Try varying each in turn and re-plot.
>
> 1. Re-run with `n.neighbors = 5` and `n.neighbors = 100`. How does each emphasise local vs. global structure?
> 2. Re-run with `dims = 1:5` and `dims = 1:30`. What does input dimensionality control?

In [ ]:
# Your code here — Part 1: RunUMAP with n.neighbors = 5 and n.neighbors = 100

<details>
<summary><strong>Exercise 1.6.1 — solution</strong></summary>

A Seurat object can hold multiple UMAP reductions side-by-side via `reduction.name`, so we don't need to rebuild the object:

```r
seu <- Seurat::RunUMAP(seu, dims = 1:20, n.neighbors = 5,
                       reduction.name = 'umap_n5',   verbose = FALSE)
seu <- Seurat::RunUMAP(seu, dims = 1:20, n.neighbors = 100,
                       reduction.name = 'umap_n100', verbose = FALSE)

Seurat::DimPlot(seu, reduction = 'umap_n5',   group.by = 'time', pt.size = 0.4) +
  ggplot2::ggtitle('n.neighbors = 5')
Seurat::DimPlot(seu, reduction = 'umap_n100', group.by = 'time', pt.size = 0.4) +
  ggplot2::ggtitle('n.neighbors = 100')
```

Small `n.neighbors` emphasises local structure (more, smaller islands; noisier layout). Large `n.neighbors` emphasises global structure (fewer, broader clusters; closer to a PCA-like view).

</details>

In [ ]:
# Your code here — Part 2: RunUMAP with dims = 1:5 and dims = 1:30

<details>
<summary><strong>Exercise 1.6.2 — solution</strong></summary>

```r
seu <- Seurat::RunUMAP(seu, dims = 1:5,  reduction.name = 'umap_d5',  verbose = FALSE)
seu <- Seurat::RunUMAP(seu, dims = 1:30, reduction.name = 'umap_d30', verbose = FALSE)

Seurat::DimPlot(seu, reduction = 'umap_d5',  group.by = 'time', pt.size = 0.4) +
  ggplot2::ggtitle('dims = 1:5')
Seurat::DimPlot(seu, reduction = 'umap_d30', group.by = 'time', pt.size = 0.4) +
  ggplot2::ggtitle('dims = 1:30')
```

`dims` controls how many PCs feed into the neighbour graph. With only 5 PCs, UMAP sees a low-resolution summary — fine for the dominant axis, but it discards the variation captured by later PCs. With 30, more biology and more noise both enter. The scree plot from Exercise 1.2 is a good guide — pick `dims` up to where the variance curve flattens.

</details>

## 1.4 Clustering and fiber-type annotation

The UMAP shows two distinct populations. If you are familiar with muscle physiology, you may remember that human muscle fibers can be broadly classified into two fiber types, slow (type 1) or fast (type 2). Each fiber type has distinct metabolic and contractile properties:

| Marker | Fiber type |
|---|---|
| MYH7 | Type I (slow oxidative) |
| MYH2 | Type IIa (fast oxidative/glycolytic) |

To explore whether our two UMAP populations correspond to the known muscle fiber types, we can extract the UMAP clusters that make the two populations and visualize the expression of known fiber type markers. If we divide this into tasks it would be: (1) hard-cluster the fibers on the kNN graph we already built, (2) label each cluster as **fast** or **slow** based on the dominant myosin heavy chain, and (3) carry that label downstream into our analysis.

In [ ]:
seu <- Seurat::FindNeighbors(seu, dims = 1:20, verbose = FALSE)
seu <- Seurat::FindClusters(seu, resolution = 0.5, verbose = FALSE)
Seurat::DimPlot(seu, reduction = 'umap', group.by = 'seurat_clusters', pt.size = 0.4, label = TRUE) +
  ggplot2::ggtitle('UMAP clusters')

Seurat::FeaturePlot(seu, features = c('MYH7', 'MYH2'), reduction = 'umap', pt.size = 0.4, order = TRUE) &
  ggplot2::scale_colour_viridis_c()

cluster_myh <- tibble::tibble(
    fiber_id = colnames(seu),
    cluster  = as.character(seu$seurat_clusters),
    MYH7     = intensities['MYH7', colnames(seu)],
    MYH2     = intensities['MYH2', colnames(seu)]
  ) |>
  group_by(cluster) |>
  summarise(MYH7 = mean(MYH7, na.rm = TRUE), MYH2 = mean(MYH2, na.rm = TRUE),
            n = dplyr::n(), .groups = 'drop') |>
  mutate(fiber_type = ifelse(MYH2 > MYH7, 'fast', 'slow'))

cluster_myh

fiber_type_map   <- setNames(cluster_myh$fiber_type, cluster_myh$cluster)
seu$fiber_type   <- unname(fiber_type_map[as.character(seu$seurat_clusters)])
metadata <- metadata |>
  left_join(tibble::tibble(fiber_id = colnames(seu), fiber_type = seu$fiber_type), by = 'fiber_id')

Seurat::DimPlot(seu, reduction = 'umap', group.by = 'fiber_type', pt.size = 0.4) +
  ggplot2::ggtitle('UMAP coloured by fiber type')

> **Exercise 1.7:** ACTN3 is a well-established fast-fiber marker. Use `FeaturePlot` to confirm that the clusters labelled `"fast"` show elevated ACTN3.



In [ ]:
# Your code here — FeaturePlot for ACTN3

<details>
<summary><strong>Exercise 1.7 — solution</strong></summary>

```r
Seurat::FeaturePlot(seu, features = 'ACTN3', reduction = 'umap',
                    pt.size = 0.4, order = TRUE) +
  ggplot2::scale_colour_viridis_c()
```

</details>

## 1.5 Differential abundance in embedding space — MILO

So far we have projected fibers into a low-dimensional space (PCA) and visualised their layout (UMAP). The next question is: **does the distribution of fibers across that space change between `pre` and `post`?** If the intervention shifts fibers from one proteomic state to another, some regions of the UMAP will be more densely populated after exercise and others less so. Detecting this is called **differential abundance (DA)** analysis.

### Why neighbourhoods instead of clusters?

The naive approach is to assign every fiber to a cluster label and count how many fibers per condition fall in each cluster. The problem is that hard clustering forces a binary decision: a fiber sitting on the boundary between two populations gets assigned to one and ignored by the other.

**MILO** avoids this by working directly on the kNN graph we already built for the UMAP. Instead of discrete clusters, it samples a set of **neighbourhoods** — each neighbourhood is the set of *k* nearest neighbors of a randomly chosen *index* fiber, forming a small, overlapping local region of the embedding. For each neighbourhood MILO counts how many fibers from each biological sample fall inside it, producing a **count matrix** (neighbourhoods × samples). This is then tested with a negative-binomial GLM — conceptually the same as what edgeR or DESeq2 do for gene counts, but applied to neighbourhood occupancy counts instead.

Key advantages over cluster-based DA:

- Fibers on population boundaries contribute to multiple overlapping neighbourhoods — no information is discarded at an arbitrary cluster edge.
- Results are mapped directly back onto the UMAP: each neighbourhood is drawn as a dot colored by its log-fold change and significance.

### Defining the biological sample

MILO counts *samples*, not individual fibers, so we need to define what constitutes one sample. With paired data and two fiber types we define each biological unit as a unique `subject × time × fiber_type` combination — 8 subjects × 2 time points × 2 fiber types = **32 samples** in total.

### Three contrasts

With fiber-type labels available we run three complementary tests:

| Model object | Design formula | Contrast | Question |
|---|---|---|---|
| `da_time` | `~ subject_id + fiber_type + time` | implicit: `post − pre` | Overall time effect, averaged across both fiber types |
| `da_fast` | `~ 0 + group + subject_id` | `groupfast_post − groupfast_pre` | `post − pre` within fast fibers only |
| `da_slow` | `~ 0 + group + subject_id` | `groupslow_post − groupslow_pre` | `post − pre` within slow fibers only |

![MILO workflow](https://raw.githubusercontent.com/fpm-cbmr/DDEA_proteomics_course/master/images/MILO_workflow.png)

*Neighbourhood-based differential abundance testing. Boxes with rounded corners are methods; rectangles are data objects.*

In [ ]:
stopifnot('fiber_type' %in% colnames(seu@meta.data))

md <- metadata |>
  mutate(sample_id = paste(subject_id, time, fiber_type, sep = '_')) |>
  as.data.frame()
rownames(md) <- md$fiber_id
imp_mat <- as.matrix(imp)[, md$fiber_id]

sce <- SingleCellExperiment::SingleCellExperiment(
  assays  = list(logcounts = imp_mat),
  colData = md
)
SingleCellExperiment::reducedDim(sce, 'PCA')  <- pca_res$x[colnames(sce), 1:30]
SingleCellExperiment::reducedDim(sce, 'UMAP') <- Seurat::Embeddings(seu, 'umap')[colnames(sce), ]

milo_obj <- miloR::Milo(sce)
milo_obj <- miloR::buildGraph(milo_obj,  k = 30, d = 20, reduced.dim = 'PCA')
milo_obj <- miloR::makeNhoods(milo_obj,  prop = 0.1, k = 30, d = 20, refined = TRUE, reduced_dims = 'PCA')
milo_obj <- miloR::countCells(milo_obj,  meta.data = md, sample = 'sample_id')
milo_obj <- miloR::buildNhoodGraph(milo_obj)

design <- md |> distinct(sample_id, subject_id, time, fiber_type) |> as.data.frame()
rownames(design) <- design$sample_id
design$time       <- factor(design$time,       levels = c('pre',  'post'))
design$fiber_type <- factor(design$fiber_type, levels = c('fast', 'slow'))
design$group      <- factor(paste(design$fiber_type, design$time, sep = '_'),
                             levels = c('fast_pre', 'fast_post', 'slow_pre', 'slow_post'))

da_time <- miloR::testNhoods(milo_obj, design = ~ subject_id + fiber_type + time, design.df = design)

da_fast <- miloR::testNhoods(milo_obj, design = ~ 0 + group + subject_id, design.df = design,
                              model.contrasts = 'groupfast_post - groupfast_pre')

da_slow <- miloR::testNhoods(milo_obj, design = ~ 0 + group + subject_id, design.df = design,
                              model.contrasts = 'groupslow_post - groupslow_pre')

da_counts <- tibble::tibble(
  fiber_type = c('fast', 'slow'),
  n_total    = c(nrow(da_fast), nrow(da_slow)),
  n_sig      = c(sum(da_fast$SpatialFDR < 0.1, na.rm = TRUE), sum(da_slow$SpatialFDR < 0.1, na.rm = TRUE)),
  n_up       = c(sum(da_fast$SpatialFDR < 0.1 & da_fast$logFC > 0, na.rm = TRUE),
                 sum(da_slow$SpatialFDR < 0.1 & da_slow$logFC > 0, na.rm = TRUE)),
  n_down     = c(sum(da_fast$SpatialFDR < 0.1 & da_fast$logFC < 0, na.rm = TRUE),
                 sum(da_slow$SpatialFDR < 0.1 & da_slow$logFC < 0, na.rm = TRUE))
)
da_counts

da_counts |>
  tidyr::pivot_longer(c(n_up, n_down), names_to = 'direction', values_to = 'n') |>
  mutate(direction = recode(direction, n_up = 'post > pre', n_down = 'pre > post')) |>
  ggplot(aes(fiber_type, n, fill = direction)) +
  geom_col(position = 'dodge') + theme_minimal() +
  labs(x = 'Fiber type', y = 'Significant neighbourhoods (SpatialFDR < 0.1)', fill = NULL)

miloR::plotNhoodGraphDA(milo_obj, da_time, layout = 'UMAP', alpha = 0.1) + ggplot2::ggtitle('Main time effect')
miloR::plotNhoodGraphDA(milo_obj, da_fast, layout = 'UMAP', alpha = 0.1) + ggplot2::ggtitle('post vs pre — fast')
miloR::plotNhoodGraphDA(milo_obj, da_slow, layout = 'UMAP', alpha = 0.1) + ggplot2::ggtitle('post vs pre — slow')

> **Exercise 1.8:** Look at `da_counts` and the bar chart. Which fiber type has more significant neighbourhoods? Are changes directionally consistent or balanced? Cross-check against the MILO maps — do significant neighbourhoods for the fast contrast sit in the fast-dominant UMAP region?

<details>
<summary><strong>Exercise 1.8 — possible answer</strong></summary>

If one fiber type has many more significant neighbourhoods, it is more responsive to the perturbation. Pay attention to direction: `post > pre` vs `pre > post` tells you whether the fiber population is expanding or contracting in that state. The MILO maps make the spatial specificity visible — if the fast contrast only lights up the fast-dominant region, the test is working correctly.

</details>

---

# Stage 2: Visualizing Differential Abundance Results

Stage 1 gave us a map of fiber-state space and identified which neighbourhoods shift between `pre` and `post`. Stage 2 zooms to the protein level: which specific proteins drive those shifts, how large and consistent are the changes, and do they behave the same in both fiber types?

Results come from a **GAM fitted per protein**, using PC1 as a continuous fiber-type proxy and testing a `time × PC1` interaction, blocking on subject. Focus on what each output column means:

| Column | Question |
|---|---|
| `Estimate_time1` | Effect size (log2FC) of `post` vs `pre`, averaged over PC1 |
| `Pr(>|t|)_time1` | Is the protein differentially abundant between `pre` and `post`? |
| `p_val_fiber_type` | Does the protein vary along PC1 (i.e. between fiber types)? |
| `p_val_interaction` | Does the time effect depend on PC1 (fiber-type-specific response)? |

In [ ]:
model_list <- readRDS('results/model_results.rds')

da_results <- dplyr::bind_rows(lapply(model_list, `[[`, 'model_results')) |>
  dplyr::mutate(
    adj_p_time        = p.adjust(`Pr(>|t|)_time1`,  method = 'BH'),
    adj_p_fiber_type  = p.adjust(p_val_fiber_type,  method = 'BH'),
    adj_p_interaction = p.adjust(p_val_interaction, method = 'BH')
  )

head(da_results)

da_results |> summarise(
  n_total       = dplyr::n(),
  n_time        = sum(adj_p_time        < 0.05, na.rm = TRUE),
  n_fiber_type  = sum(adj_p_fiber_type  < 0.05, na.rm = TRUE),
  n_interaction = sum(adj_p_interaction < 0.05, na.rm = TRUE),
  n_time_up     = sum(adj_p_time < 0.05 & Estimate_time1 > 0, na.rm = TRUE),
  n_time_down   = sum(adj_p_time < 0.05 & Estimate_time1 < 0, na.rm = TRUE)
)

## 2.1 Boxplots of single features

The first sanity check: look at the raw intensities of a few top hits, stratified by `time`.

In [ ]:
feature <- da_results |> arrange(`Pr(>|t|)_time1`) |> dplyr::slice(1) |> pull(gene)

plot_df <- tibble(fiber_id = colnames(intensities), intensity = intensities[feature, ]) |>
  left_join(metadata, by = 'fiber_id')

ggplot(plot_df, aes(time, intensity, fill = time)) +
  geom_boxplot(outlier.shape = NA, alpha = 0.6) +
  geom_jitter(width = 0.2, size = 0.3, alpha = 0.3) +
  theme_minimal() + labs(title = feature, y = 'log2 intensity')

> **Exercise 2.1:**
>
> 1. Pick a protein with `adj_p_time > 0.5` and reproduce the boxplot. Does the lack of a shift match the test?
> 2. Redraw the top time hit (`S100A13`) coloured by `subject_id`, with one line per subject connecting `pre` and `post` means. Is the shift consistent across subjects?

In [ ]:
# Your code here — Part 1: boxplot of a non-significant protein

<details>
<summary><strong>Exercise 2.1.1 — solution</strong></summary>

```r
nonsig_feature <- da_results %>%
  filter(adj_p_time > 0.5) %>%
  dplyr::slice(1) %>%
  pull(gene)

plot_df <- tibble(
  fiber_id  = colnames(intensities),
  intensity = intensities[nonsig_feature, ]
) %>%
  left_join(metadata, by = 'fiber_id')

ggplot(plot_df, aes(time, intensity, fill = time)) +
  geom_boxplot(outlier.shape = NA, alpha = 0.6) +
  geom_jitter(width = 0.2, size = 0.3, alpha = 0.3) +
  theme_minimal() +
  labs(title = nonsig_feature, y = 'log2 intensity')
```

Compared to a top hit, the two boxes should overlap heavily and the medians sit on essentially the same line.

</details>

In [ ]:
# Your code here — Part 2: per-subject line plot for S100A13

<details>
<summary><strong>Exercise 2.1.2 — solution</strong></summary>

```r
feature <- 'S100A13'

plot_df <- tibble(
  fiber_id  = colnames(intensities),
  intensity = intensities[feature, ]
) %>%
  left_join(metadata, by = 'fiber_id') %>%
  group_by(subject_id, time) %>%
  summarise(intensity = mean(intensity, na.rm = TRUE), .groups = 'drop')

ggplot(plot_df, aes(time, intensity, group = subject_id, colour = subject_id)) +
  geom_line(linewidth = 0.8) +
  geom_point(size = 2) +
  theme_minimal() +
  labs(title = feature, y = 'log2 intensity (subject mean)')
```

If all 8 lines slope in the same direction, the effect is consistent. If a couple of subjects buck the trend, the population estimate is being pulled by a subset.

</details>

## 2.2 Volcano plots

A volcano plot shows effect size (`Estimate_time1`, the log2FC of `post − pre`) on the x-axis against statistical significance (`-log10 p-value`) on the y-axis. It is the standard one-glance summary of a DA analysis. I particularly like producing an interactive volcano so I can decide which proteins I will assign a label in my final static volcano for publication.

In [ ]:
volcano_df <- da_results |>
  mutate(sig   = factor(ifelse(adj_p_time < 0.05, 'adj_p < 0.05', 'n.s.'), levels = c('n.s.', 'adj_p < 0.05')),
         label = paste0('gene: ', gene, '<br>log2FC: ', signif(Estimate_time1, 3), '<br>adj_p: ', signif(adj_p_time, 3)))

p_v <- ggplot(volcano_df, aes(x = Estimate_time1, y = -log10(`Pr(>|t|)_time1`), colour = sig, text = label)) +
  geom_point(alpha = 0.6, size = 1.2) +
  geom_hline(yintercept = -log10(0.05), linetype = 'dashed', colour = 'grey60') +
  geom_vline(xintercept = 0, linetype = 'dashed', colour = 'grey60') +
  scale_colour_manual(values = c('n.s.' = 'grey70', 'adj_p < 0.05' = 'firebrick')) +
  theme_minimal() + labs(x = 'log2FC (post − pre)', y = '-log10(p-value)', colour = NULL)
p_v
plotly::ggplotly(p_v, tooltip = 'text')

> **Exercise 2.2:**
>
> 1. How many proteins are significant at `adj_p_time < 0.05`? Is the cloud symmetric? Hover the interactive version to identify top up- and down-regulated genes.
> 2. Redraw using **PC1 loading vs. fiber-type p-value** (`pca_res$rotation[, "PC1"]` on x, `p_val_fiber_type` on y). More or fewer hits?

In [ ]:
# Your code here — PC1-loading volcano (join pca_res$rotation onto da_results)

<details>
<summary><strong>Exercise 2.2.2 — solution</strong></summary>

```r
pc1_load <- tibble(
  gene        = rownames(pca_res$rotation),
  pc1_loading = pca_res$rotation[, 'PC1']
)

volcano_pc1_df <- da_results %>%
  left_join(pc1_load, by = 'gene') %>%
  mutate(
    sig   = factor(ifelse(adj_p_fiber_type < 0.05, 'adj_p < 0.05', 'n.s.'),
                   levels = c('n.s.', 'adj_p < 0.05')),
    label = paste0('gene: ', gene,
                   '<br>PC1 loading: ', signif(pc1_loading, 3),
                   '<br>adj_p: ',       signif(adj_p_fiber_type, 3))
  )

p_volcano_pc1 <- ggplot(
    volcano_pc1_df,
    aes(x = pc1_loading, y = -log10(p_val_fiber_type),
        colour = sig, text = label)
  ) +
  geom_point(alpha = 0.6, size = 1.2) +
  geom_hline(yintercept = -log10(0.05), linetype = 'dashed', colour = 'grey60') +
  geom_vline(xintercept = 0,            linetype = 'dashed', colour = 'grey60') +
  scale_colour_manual(values = c('n.s.' = 'grey70', 'adj_p < 0.05' = 'firebrick')) +
  theme_minimal() +
  labs(x = 'PC1 loading', y = '-log10(adj. p-value)', colour = NULL,
       title = 'Fiber-type axis (PC1) vs. GAM p-value')

plotly::ggplotly(p_volcano_pc1, tooltip = 'text')
```

The fiber-type volcano shows many more hits than the time volcano — PC1 is the dominant axis in the data, so almost every protein co-varies with it to some degree. Also, this plot condenses the behavior of a protein across PC1 into a single data point, so a heatmap of associated features across PC1 or feature plots on the UMAP would be more informative.

</details>

## 2.3 Feature plots on UMAP

Overlaying a single protein's intensity onto the UMAP embedding shows *where* in fiber-state space that protein is high or low — often more informative than a volcano plot or boxplot when the contrast is driven by a sub-population.

In [ ]:
Seurat::FeaturePlot(seu, features = 'MYH7', reduction = 'umap', pt.size = 0.4, order = TRUE) +
  ggplot2::scale_colour_viridis_c()

> **Exercise 2.3:** Reproduce the feature plot for the top time hit (`S100A13`) and the top interaction hit (`ALDH1B1`). Does the interaction-driven protein localise differently?

In [ ]:
# Your code here — FeaturePlot for S100A13 and ALDH1B1

<details>
<summary><strong>Exercise 2.3 — solution</strong></summary>

```r
Seurat::FeaturePlot(seu, features = 'S100A13', reduction = 'umap',
                    pt.size = 0.4, order = TRUE) +
  ggplot2::scale_colour_viridis_c()

Seurat::FeaturePlot(seu, features = 'ALDH1B1', reduction = 'umap',
                    pt.size = 0.4, order = TRUE) +
  ggplot2::scale_colour_viridis_c()
```

`S100A13`'s effect is largely fiber-type-agnostic — it is difficult to visualize the changes in intensity because of the scale. `ALDH1B1` is both a top time hit **and** a top interaction hit — the time-driven shift is concentrated in one fiber-type region rather than spread uniformly. This is the visual signature of a `time × fiber_type` interaction.

</details>

## 2.4 Association of protein intensity with PC1 (fiber type)

Because PC1 is the dominant axis of variation in this dataset and is strongly associated with fiber type (slow vs. fast), `p_val_fiber_type` is essentially asking: *does this protein vary systematically between slow and fast fibers?* Proteins with a low p-value here are the molecular markers of fiber identity. Visualizing intensity against PC1 as a continuous axis is more informative than a boxplot split by cluster label, because it shows the full gradient rather than a discretised summary.

In [ ]:
feature <- da_results |> arrange(p_val_fiber_type) |> dplyr::slice(1) |> pull(gene)
tibble(fiber_id = colnames(intensities), intensity = intensities[feature, ]) |>
  left_join(metadata, by = 'fiber_id') |>
  ggplot(aes(PC1, intensity)) +
  geom_point(size = 0.4, alpha = 0.4) +
  geom_smooth(method = 'gam', formula = y ~ s(x), se = TRUE) +
  theme_minimal() + labs(title = feature, x = 'PC1 (fiber-type axis)', y = 'log2 intensity')

> **Exercise 2.4:** Reproduce the scatter for `C4orf54` (high `p_val_fiber_type`) vs `GATD3B` (top hit). Compute Pearson correlation with PC1 for both to confirm.

In [ ]:
# Your code here — scatter plots and Pearson correlations for GATD3B and C4orf54

<details>
<summary><strong>Exercise 2.4 — solution</strong></summary>

```r
plot_one <- function(gene) {
  plot_df <- tibble(
    fiber_id  = colnames(intensities),
    intensity = intensities[gene, ]
  ) %>%
    left_join(metadata, by = 'fiber_id')

  ggplot(plot_df, aes(PC1, intensity)) +
    geom_point(size = 0.4, alpha = 0.4) +
    geom_smooth(method = 'gam', formula = y ~ s(x), se = TRUE) +
    theme_minimal() +
    labs(title = gene, x = 'PC1', y = 'log2 intensity')
}

plot_one('GATD3B')
plot_one('C4orf54')

cor_with_pc1 <- function(gene) {
  vals <- intensities[gene, ]
  ok   <- !is.na(vals)
  cor(vals[ok], metadata$PC1[ok])
}

c(GATD3B  = cor_with_pc1('GATD3B'),
  C4orf54 = cor_with_pc1('C4orf54'))
```

`GATD3B` shows a clean monotonic gradient with PC1; `C4orf54`'s cloud is flat. `GATD3B` has a much higher |cor| than `C4orf54`, which is close to 0.

</details>

## 2.5 Interaction between PC1 and time

`p_val_interaction` flags proteins where the **size or direction of the pre→post shift depends on where a fiber sits on the PC1 axis** — in other words, the exercise response is not uniform across fiber types. A protein with a strong main time effect but no interaction changes by roughly the same amount in slow and fast fibers. A protein with a significant interaction changes a lot in one fiber type and barely moves in the other (or even shifts in opposite directions).

The cleanest way to see this is to plot intensity against PC1 separately for `pre` and `post` and fit a GAM smoother to each. Parallel curves signal a pure time effect; diverging or crossing curves signal an interaction.

In [ ]:
plot_interaction <- function(gene) {
  tibble(fiber_id = colnames(intensities), intensity = intensities[gene, ]) |>
    left_join(metadata, by = 'fiber_id') |>
    ggplot(aes(PC1, intensity, colour = time, fill = time)) +
    geom_point(size = 0.4, alpha = 0.3) +
    geom_smooth(method = 'gam', formula = y ~ s(x), se = TRUE, alpha = 0.2) +
    theme_minimal() + labs(title = gene, x = 'PC1 (fiber-type axis)', y = 'log2 intensity')
}

feature <- da_results |> arrange(p_val_interaction) |> dplyr::slice(1) |> pull(gene)
plot_interaction(feature)

> **Exercise 2.5:** Plot `S100A13` (strong time effect, no interaction) and `ALDH1B1` (top interaction hit). Are the `S100A13` smoothers parallel? Do the `ALDH1B1` smoothers cross or diverge?

In [ ]:
# Your code here — plot_interaction('S100A13') and plot_interaction('ALDH1B1')

<details>
<summary><strong>Exercise 2.5 — solution</strong></summary>

```r
plot_interaction <- function(gene) {
  plot_df <- tibble(
    fiber_id  = colnames(intensities),
    intensity = intensities[gene, ]
  ) %>%
    left_join(metadata, by = 'fiber_id')

  ggplot(plot_df, aes(PC1, intensity, colour = time, fill = time)) +
    geom_point(size = 0.4, alpha = 0.3) +
    geom_smooth(method = 'gam', formula = y ~ s(x), se = TRUE, alpha = 0.2) +
    theme_minimal() +
    labs(title = gene, x = 'PC1', y = 'log2 intensity')
}

plot_interaction('S100A13')
plot_interaction('ALDH1B1')
```

For a pure time effect (`S100A13`), the two smoothers should sit at different y-levels but track the same shape — vertically shifted but **parallel**. For `ALDH1B1`, the smoothers should be clearly **non-parallel** — the gap between pre and post widens or narrows along PC1. A significant interaction means the `post − pre` change is not uniform across fibers — one fiber-type tail responds more than the other. If the curves *cross*, the direction of the time effect flips between fiber types; if they only *diverge*, one fiber type responds and the other barely does.

</details>

---

# Stage 3: Heatmaps, Co-Regulated Modules & Pathway Enrichment

The volcano and scatter plots from Stage 2 tell you *which* proteins change and how large the effect is — but they treat every protein independently. Stage 3 asks a different question: do the proteins that change do so **in a coordinated way**? Proteins that are co-regulated across fibers tend to share upstream regulators, belong to the same complex, or participate in the same biological process. Revealing that structure requires visualising many proteins at once, which is what heatmaps are built for.

In [ ]:
sig_proteins <- da_results |> filter(adj_p_time < 0.05) |> pull(gene)
mat  <- intensities[sig_proteins, ]
mat  <- t(scale(t(mat)))

ann_col <- metadata |> dplyr::select(fiber_id, time, PC1) |> tibble::column_to_rownames('fiber_id')
ann_col <- ann_col[colnames(mat), , drop = FALSE]

quantiles <- quantile(mat, probs = c(0.01, 0.5, 0.99), na.rm = TRUE)
breaks <- c(seq(quantiles[1], 0, length.out = 50), seq(0, quantiles[3], length.out = 51)[-1])
colors <- c(colorRampPalette(c('darkblue', 'white'))(50), colorRampPalette(c('white', 'red'))(50))

ph <- pheatmap::pheatmap(mat, annotation_col = ann_col, color = colors, breaks = breaks,
  border_color = NA, show_rownames = FALSE, show_colnames = FALSE,
  cluster_rows = TRUE, cluster_cols = TRUE, cutree_rows = 6, scale = 'none', fontsize = 8)
ggplotify::as.ggplot(ph)

The dominant structure is the **fiber-type gradient** (PC1), not the pre/post contrast — expected, because fiber type explains far more variance. The `diff_smooth` heatmap below fixes this by using the GAM-corrected time effect directly.

In [ ]:
tmp   <- model_list[sig_proteins]
diffs <- map(tmp, 'diff_smooth') |>
  imap(\(df, gene) df |> mutate(gene = gene)) |>
  bind_rows()

diff_mat <- diffs |>
  dplyr::select(gene, PC1, .diff) |>
  mutate(PC1_bin = cut(PC1, breaks = 25), .diff = .diff * -1) |>
  dplyr::select(gene, PC1_bin, .diff) |>
  group_by(gene, PC1_bin) |>
  summarise(mean_diff = mean(.diff, na.rm = TRUE), .groups = 'drop') |>
  pivot_wider(names_from = PC1_bin, values_from = mean_diff) |>
  rowwise() |>
  dplyr::filter(max(c_across(where(is.numeric)), na.rm = TRUE) >= 0.5 |
                min(c_across(where(is.numeric)), na.rm = TRUE) <= -0.5) |>
  ungroup() |> column_to_rownames('gene') |> t() |> t() |> as.data.frame()

bin_midpoints <- colnames(diff_mat) |>
  stringr::str_extract_all('-?[0-9]+\\.?[0-9]*') |>
  purrr::map_dbl(\(x) mean(as.numeric(x)))
ann_diff <- data.frame(PC1 = bin_midpoints, row.names = colnames(diff_mat))

q_diff <- quantile(diff_mat, probs = c(0.01, 0.99), na.rm = TRUE)
breaks_diff <- c(seq(q_diff[1], 0, length.out = 50), seq(0, q_diff[2], length.out = 51)[-1])
colors_diff <- c(colorRampPalette(c('darkblue', 'white'))(50), colorRampPalette(c('white', 'red'))(50))

ph_diff <- pheatmap::pheatmap(diff_mat, annotation_col = ann_diff, color = colors_diff,
  breaks = breaks_diff, border_color = NA, show_rownames = FALSE, show_colnames = FALSE,
  cluster_rows = TRUE, cluster_cols = FALSE, scale = 'none', fontsize = 8,
  main = 'GAM smooth: post − pre difference across PC1')
ggplotify::as.ggplot(ph_diff)

## 3.2 Extracting module membership with navmix

navmix is a probabilistic model-based clustering method. We run it on `diff_mat` (after removing NA rows that would cause a row-count mismatch) and use the BIC to select K.

In [ ]:
diff_mat_clean <- diff_mat[complete.cases(diff_mat), ]

set.seed(42)
cl_bic <- navmix(diff_mat_clean, K = 10, plot = TRUE, plot_radial = TRUE)
plot(1:10, cl_bic$BIC, xlab = 'K', ylab = 'BIC', main = 'BIC — select K at the elbow')

clusters_heatmap <- navmix(diff_mat_clean, K = 5, plot = TRUE, plot_radial = TRUE)

tmp.cl <- data.frame(gene = rownames(diff_mat_clean), cluster = clusters_heatmap$fit$z)

row_ann  <- tmp.cl |> tibble::column_to_rownames('gene') |> dplyr::mutate(cluster = factor(cluster))
row_order <- order(tmp.cl$cluster)
diff_mat_plot <- diff_mat_clean[row_order, ]

ph_clusters <- pheatmap::pheatmap(diff_mat_plot,
  annotation_row = row_ann, annotation_col = ann_diff,
  color = colors_diff, breaks = breaks_diff, border_color = NA,
  show_rownames = FALSE, show_colnames = FALSE,
  cluster_rows = FALSE, cluster_cols = FALSE, scale = 'none', fontsize = 8,
  main = 'GAM smooth: post − pre across PC1 (navmix clusters)')
ggplotify::as.ggplot(ph_clusters)

## 3.3 Pathway enrichment per module (ORA)

ORA asks: are GO Biological Process terms annotated to more proteins in this module than expected by chance? Key choices: set the **universe** to `rownames(intensities)` (proteins we actually measured), use `keyType = 'SYMBOL'`, and correct with BH.

In [ ]:
tmp.cl |> dplyr::count(cluster, name = 'n_proteins')

In [ ]:
module_of_interest <- 1   # change after inspecting the heatmap

module_genes <- tmp.cl |> dplyr::filter(cluster == module_of_interest) |> dplyr::pull(gene)
cat('Module', module_of_interest, '—', length(module_genes), 'proteins\n')

ora_res <- clusterProfiler::enrichGO(
  gene = module_genes, universe = rownames(intensities),
  OrgDb = org.Hs.eg.db::org.Hs.eg.db, keyType = 'SYMBOL',
  ont = 'BP', pAdjustMethod = 'BH', pvalueCutoff = 0.05, qvalueCutoff = 0.2
)

ora_df <- as_tibble(ora_res@result) |> dplyr::filter(p.adjust < 0.05) |> arrange(p.adjust)
cat(nrow(ora_df), 'significant GO:BP terms\n')

ora_df |> slice_head(n = 20) |>
  mutate(GeneRatio_num = purrr::map_dbl(GeneRatio, \(x) { v <- as.integer(strsplit(x, '/')[[1]]); v[1]/v[2] })) |>
  ggplot(aes(GeneRatio_num, forcats::fct_reorder(Description, GeneRatio_num),
             size = GeneRatio_num, colour = -log10(p.adjust))) +
  geom_point() +
  scale_colour_gradient(low = 'steelblue', high = 'firebrick') +
  scale_size_continuous(range = c(2, 7)) +
  theme_minimal() +
  labs(title = paste0('GO:BP ORA — module ', module_of_interest, ' (', length(module_genes), ' proteins)'),
       x = 'Gene ratio', y = NULL, colour = '-log10(adj.p)', size = 'Gene ratio')

The enriched terms tell you **what the module is doing biologically**. The heatmap pattern tells you *when and where*; the ORA tells you *what*.

> **Exercise 3.1:** Repeat the full pipeline (diff_smooth heatmap → navmix → ORA) using **interaction-significant proteins** (`adj_p_interaction < 0.05`) instead of time-significant ones. Do the modules reveal a different biological story?

In [ ]:
# Your code here — filter interaction_proteins, rebuild diff_mat, navmix, ORA

<details>
<summary><strong>Exercise 3.1 — possible answer</strong></summary>

The code follows the same pipeline as sections 3.1–3.3, just starting from a different protein list. Key variables use `_int` suffixes to avoid overwriting the time-based objects.

```r
interaction_proteins <- da_results |>
  dplyr::filter(adj_p_interaction < 0.05) |>
  dplyr::pull(gene)

cat(length(interaction_proteins), 'proteins with significant PC1 × time interaction\n')

tmp_int <- model_list[interaction_proteins]

diffs_int <- map(tmp_int, 'diff_smooth') |>
  imap(\(df, gene) df |> mutate(gene = gene)) |>
  bind_rows()

diff_mat_int <- diffs_int |>
  dplyr::select(gene, PC1, .diff) |>
  mutate(PC1_bin = cut(PC1, breaks = 25), .diff = .diff * -1) |>
  dplyr::select(gene, PC1_bin, .diff) |>
  group_by(gene, PC1_bin) |>
  summarise(mean_diff = mean(.diff, na.rm = TRUE), .groups = 'drop') |>
  pivot_wider(names_from = PC1_bin, values_from = mean_diff) |>
  rowwise() |>
  dplyr::filter(max(c_across(where(is.numeric)), na.rm = TRUE) >= 0.5 |
                min(c_across(where(is.numeric)), na.rm = TRUE) <= -0.5) |>
  ungroup() |>
  column_to_rownames('gene') |>
  t() |> t() |>
  as.data.frame()

bin_mid_int <- colnames(diff_mat_int) |>
  stringr::str_extract_all('-?[0-9]+\\.?[0-9]*') |>
  purrr::map_dbl(\(x) mean(as.numeric(x)))

ann_int    <- data.frame(PC1 = bin_mid_int, row.names = colnames(diff_mat_int))
q_int      <- quantile(diff_mat_int, probs = c(0.01, 0.99), na.rm = TRUE)
breaks_int <- c(seq(q_int[1], 0, length.out = 50), seq(0, q_int[2], length.out = 51)[-1])
colors_int <- c(colorRampPalette(c('darkblue', 'white'))(50),
                colorRampPalette(c('white', 'red'))(50))

diff_mat_int_clean <- diff_mat_int[complete.cases(diff_mat_int), ]
set.seed(42)
cl_int <- navmix(diff_mat_int_clean, K = 5, plot = TRUE, plot_radial = TRUE)

tmp.cl_int   <- data.frame(gene = rownames(diff_mat_int_clean), cluster = cl_int$fit$z)
row_ann_int  <- tmp.cl_int |> tibble::column_to_rownames('gene') |> dplyr::mutate(cluster = factor(cluster))
diff_mat_int_pl <- diff_mat_int_clean[order(tmp.cl_int$cluster), ]

pheatmap::pheatmap(diff_mat_int_pl,
  annotation_row = row_ann_int, annotation_col = ann_int,
  color = colors_int, breaks = breaks_int, border_color = NA,
  show_rownames = FALSE, show_colnames = FALSE,
  cluster_rows = FALSE, cluster_cols = FALSE, scale = 'none', fontsize = 8,
  main = 'Interaction proteins — navmix clusters') |>
  ggplotify::as.ggplot()

mod_int       <- 1
mod_genes_int <- tmp.cl_int |> dplyr::filter(cluster == mod_int) |> dplyr::pull(gene)

clusterProfiler::enrichGO(
  gene = mod_genes_int, universe = rownames(intensities),
  OrgDb = org.Hs.eg.db::org.Hs.eg.db, keyType = 'SYMBOL',
  ont = 'BP', pAdjustMethod = 'BH', pvalueCutoff = 0.05, qvalueCutoff = 0.2
) |>
  as_tibble() |>
  dplyr::filter(p.adjust < 0.05) |>
  dplyr::arrange(p.adjust) |>
  dplyr::slice_head(n = 20) |>
  dplyr::mutate(GeneRatio_num = purrr::map_dbl(GeneRatio,
    \(x) { v <- as.integer(strsplit(x, '/')[[1]]); v[1]/v[2] })) |>
  ggplot(aes(GeneRatio_num, forcats::fct_reorder(Description, GeneRatio_num),
             size = GeneRatio_num, colour = -log10(p.adjust))) +
  geom_point() +
  scale_colour_gradient(low = 'steelblue', high = 'firebrick') +
  scale_size_continuous(range = c(2, 7)) +
  theme_minimal() +
  labs(title = paste0('GO:BP ORA — interaction module ', mod_int), x = 'Gene ratio', y = NULL)
```

</details>

---

# Extended Exercises

These exercises are designed for students who finish early or want to go deeper. Each one extends a specific section of the tutorial — the section label is noted next to the title.

In [ ]:
# Your code here — Exercise A

### Exercise B — MILO sensitivity to neighbourhood parameters `(§1.5)`

Re-run MILO with `k = 15, prop = 0.2` (finer) and `k = 50, prop = 0.05` (coarser). For each, re-run `testNhoods` with the main-effect design and compare: how many neighbourhoods? How many significant? Do they sit in the same UMAP regions?

In [ ]:
# Your code here — Exercise B

### Exercise C — Adding an effect-size filter to the volcano `(§2.2)`

Re-colour the volcano into three categories: `"n.s."`, `"adj_p only"` (significant but `|Estimate_time1| < 0.5`), and `"sig + |FC| ≥ 0.5"`. Run ORA on the proteins in the `"adj_p only"` stratum — do they enrich for anything, or is the signal noise?

In [ ]:
# Your code here — Exercise C

### Exercise D — navmix K selection `(§3.2)`

Re-run navmix with K = 3 and K = 7 on `diff_mat_clean`. For each K, render the cluster heatmap and run ORA on every module (use `lapply` over `unique(tmp.cl$cluster)`). Do adjacent modules at higher K look like splits of a single K = 3 module, or do they capture new biology?

In [ ]:
# Your code here — Exercise D

### Exercise E — All-module enrichment `(§3.3)`

Use `purrr::map` to run `enrichGO` on every navmix module, collect the top enriched term per module, and display them in a summary table. Do the five modules tell a coherent biological story together?

In [ ]:
# Your code here — Exercise E

### Exercise F — From module to mechanism `(§3.3)`

Pick the module with the most significant GO:BP term from Exercise E. Look up its canonical upstream regulator (e.g. PGC-1α for oxidative metabolism, AMPK for energy sensing). Check `rownames(intensities)` — is it measured? If yes, plot its intensity vs PC1 (split by `time`) and look up its `adj_p_time` and `adj_p_interaction` in `da_results`. If not significant at the protein level, check its `diff_smooth` curve — does it show a consistent directional trend?

In [ ]:
# Your code here — Exercise F